# **02477 Bayesian Machine Learning | comprehensive study totes**



---

###  **Table of Contents**




1. [Week 1 | foundations & beta-binomial model](#week1)

2. [Week 2 | predictions, plug-in & grid approximations](#week2)

3. [Week 3 | bayesian classification & laplace approximations](#week3)

4. [Week 4 | bayesian linear regression](#week4)

5. [Week 5 | gaussian processes (regression)](#week5)

6. [Week 6 | GP classification & kernels](#week6)

7. [Week 7 | multi-class, decision theory & calibration](#week7)

8. [Week 8 | monte carlo & metropolis-hastings](#week8)

9. [Week 9 | advanced MCMC & convergence diagnostics](#week9)

10. [Exam cheat-sheet | key formulae reference](#cheatsheet)



---

In [ ]:
# Standard imports used throughout the notebook
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import norm, beta as beta_dist, binom, multivariate_normal
from scipy.special import expit as sigmoid  # sigmoid = logistic function

plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 4)})
print('Imports OK')

<a id='week1'></a>

<div class="alert alert-block alert-info">

### **Week 1 | foundations & beta-binomial model**

Bayesian machine learning, the binominal model, maximum likelihood estimation (MLE), bayesian inference, beta-binomial model
</div>

##### **1.1 Why bayesian machine learning?**


Classical ML picks **one** parameter setting and makes predictions with it.  

Bayesian ML **averages over all parameter settings** weighted by their posterior probability:

$$p(y^*|y) = \int p(y^*|\theta)\, p(\theta|y)\, d\theta$$

**Advantages**

> Principled uncertainty quantification (epistemic + aleatoric)

> Less prone to overfitting (prior regularises)

> Natural model selection via marginal likelihood

> Better decision-making under uncertainty


##### **1.2 Bayes' Rule - the cornerstone**


$$\underbrace{p(\theta | y)}_{\text{posterior}} = \frac{\underbrace{p(y|\theta)}_{\text{likelihood}}\; \underbrace{p(\theta)}_{\text{prior}}}{\underbrace{p(y)}_{\text{evidence/marginal likelihood}}}$$

| Term | Symbol | Meaning |
|------|--------|---------|
| Prior | $p(\theta)$ | Belief about $\theta$ **before** seeing data |
| Likelihood | $p(y\|\theta)$ | Probability of data **given** parameters |
| Posterior | $p(\theta\|y)$ | Updated belief **after** seeing data |
| Evidence | $p(y) = \int p(y\|\theta)p(\theta)d\theta$ | Normalisation constant; useful for model selection |

**Key insight:** $p(\theta|y) \propto p(y|\theta)\,p(\theta)$ - we never need to compute $p(y)$ to find the posterior *shape*.


##### **1.3 Types of uncertainty**




| Type | Cause | Reducible? |
|------|-------|-----------|
| **Epistemic** | Lack of data / knowledge | Yes - more data helps |
| **Aleatoric** | Inherent noise in the process | No - irreducible |


##### **1.4 Common distributions (building blocks)**


| Distribution | Domain | PMF / PDF | Mean | Variance |
|---|---|---|---|---|
| $\text{Bernoulli}(\mu)$ | $\{0,1\}$ | $\mu^y(1-\mu)^{1-y}$ | $\mu$ | $\mu(1-\mu)$ |
| $\text{Binomial}(N,\theta)$ | $\{0,\dots,N\}$ | $\binom{N}{y}\theta^y(1-\theta)^{N-y}$ | $N\theta$ | $N\theta(1-\theta)$ |
| $\text{Beta}(a,b)$ | $[0,1]$ | $\propto\theta^{a-1}(1-\theta)^{b-1}$ | $\frac{a}{a+b}$ | $\frac{ab}{(a+b)^2(a+b+1)}$ |
| $\mathcal{N}(\mu,\sigma^2)$ | $\mathbb{R}$ | $\frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$ | $\mu$ | $\sigma^2$ |
| $\text{Poisson}(\lambda)$ | $\{0,1,2,\dots\}$ | $e^{-\lambda}\lambda^y/y!$ | $\lambda$ | $\lambda$ |
| $\text{Categorical}(\pi)$ | $\{1,\dots,K\}$ | $\prod_k \pi_k^{[y=k]}$ | - | - |


##### **1.5 The beta-binomial model (conjugate)**



**Setup:** coin with unknown bias $\theta \in [0,1]$, observe $y$ heads in $N$ flips.

$$p(\theta) = \text{Beta}(\theta|a_0, b_0) \qquad p(y|\theta) = \text{Bin}(y|N, \theta)$$

**Posterior** (conjugacy means the posterior is in the same family as the prior):
$$p(\theta|y) = \text{Beta}(\theta \mid y + a_0,\; N - y + b_0)$$

> **Why conjugacy works:** Multiply the Beta PDF by the Binomial PMF; the $\theta$-dependent terms combine to give another Beta.

**Posterior Predictive** (probability of $y^*$ heads in $N^*$ future flips):
$$p(y^*|y, N^*) = \int_0^1 \text{Bin}(y^*|N^*, \theta)\,\text{Beta}(\theta|a_N, b_N)\,d\theta = \text{BetaBin}(y^*|N^*, a_N, b_N)$$

**Posterior Mean (Bayes estimator):**
$$\hat{\theta}_{\text{Bayes}} = \mathbb{E}[\theta|y] = \frac{y + a_0}{N + a_0 + b_0}$$

Note: as $N \to \infty$, the posterior mean approaches the MLE $y/N$. The prior only matters when data is scarce.


##### **1.6 Maximum likelihood estimation (MLE)**




$$\hat{\theta}_{\text{MLE}} = \arg\max_\theta \log p(y|\theta)$$

For the Binomial: $\hat{\theta}_{\text{MLE}} = y/N$. Problem: if $y=0$, then $\hat{\theta}=0$ — classic **overfitting** on small data.


##### **1.7 Credibility intervals versus Confidence intervals**


A **95% credibility interval** $[\ell, u]$ satisfies:
$$P(\theta \in [\ell, u] | y) = 0.95$$
This is a direct probability statement about $\theta$ — much more intuitive than a frequentist confidence interval.

In [ ]:
# Week 1 demo: beta-binomial bayesian updating

theta_grid = np.linspace(0.001, 0.999, 500)

# Prior: Beta(1,1) = Uniform
a0, b0 = 1, 1

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
datasets = [(0, 3), (1, 10), (5, 30), (17, 100)]  # (heads, total)

for ax, (y, N) in zip(axes, datasets):
    a_post = y + a0
    b_post = (N - y) + b0
    prior_pdf = beta_dist.pdf(theta_grid, a0, b0)
    post_pdf  = beta_dist.pdf(theta_grid, a_post, b_post)
    
    ax.plot(theta_grid, prior_pdf, '--', label='Prior', color='gray')
    ax.plot(theta_grid, post_pdf, label=f'Posterior (N={N}, y={y})', color='steelblue', lw=2)
    ax.axvline(y/N if N>0 else 0, color='red', linestyle=':', label='MLE')
    ax.axvline(a_post/(a_post+b_post), color='green', linestyle=':', label='Bayes mean')
    ax.set_xlabel('θ'); ax.set_title(f'N={N}, y={y}')
    ax.legend(fontsize=8)

fig.suptitle('Beta-Binomial: Effect of data on posterior', fontweight='bold')
plt.tight_layout()
plt.show()
print('As N grows, posterior concentrates near the true value and MLE ≈ Bayes mean.')

<a id='week2'></a>

<div class="alert alert-block alert-info">

### **Week 2 | predictors, plug-in & grid approximation**

Probabilistic machine learning, plug-in approximation, grid approximation, non-conjugate models, logistic regression

</div>

##### **2.1 The posterior predictive distribution**




The fully Bayesian prediction marginalises over parameters:
$$p(y^*|y) = \int p(y^*|\theta)\, p(\theta|y)\, d\theta = \mathbb{E}_{p(\theta|y)}[p(y^*|\theta)]$$

This **accounts for parameter uncertainty** — predictions are more conservative (wider uncertainty) than the plug-in.


##### **2.2 The plug-in (point-estimate) approximation**




If we assume $p(\theta|y) \approx \delta(\theta - \hat{\theta})$ (Dirac delta centred at a point estimate):
$$p(y^*|y) \approx p(y^*|\hat{\theta})$$

**Dirac delta properties:**
$$\delta(x-\mu) = 0 \text{ for } x\neq\mu, \quad \int \delta(x-\mu)dx = 1, \quad \int f(x)\delta(x-\mu)dx = f(\mu)$$

> **Danger:** Plug-in ignores parameter uncertainty → **overconfident** predictions. This is what standard deep learning does!


##### **2.3 Grid approximation**




For non-conjugate models where the posterior is intractable, discretise the parameter space.

**Algorithm:**
1. Define a grid $\theta_1 < \theta_2 < \cdots < \theta_M$
2. Evaluate unnormalised posterior: $\tilde{q}(\theta_m) = p(y|\theta_m)\,p(\theta_m)$
3. Normalise: $q(\theta_m) = \tilde{q}(\theta_m) / \sum_{j} \tilde{q}(\theta_j)$
4. Posterior summaries: $\mathbb{E}[\theta] \approx \sum_m \theta_m q(\theta_m)$

**Posterior predictive via grid:**
$$p(y^*|y) \approx \sum_m p(y^*|\theta_m)\,q(\theta_m)$$

**Limitation:** Curse of dimensionality — grid size is $M^D$ for $D$ parameters.


##### **2.4 2D Grid approximation (exam style)**


Given a 2D grid $q(w_1, w_2)$:
- **Marginal**: $q(w_1) = \sum_{w_2} q(w_1, w_2)$
- **Mean**: $\mathbb{E}[w_1] = \sum_{w_1,w_2} w_1 \cdot q(w_1, w_2)$
- **Variance**: $\mathbb{V}[w_1] = \mathbb{E}[w_1^2] - (\mathbb{E}[w_1])^2$
- **Predictive**: $p(y^*|y,x^*) \approx \sum_{w_1,w_2} p(y^*|w_1,w_2,x^*)\cdot q(w_1,w_2)$

In [ ]:
# Week 2 demo: grid approximation - 1D non-conjugate example

from scipy.stats import norm as gaussian

# Observed data
y_obs = np.array([0, 0, 1])  # 1 click out of 3 views
N, n_clicks = len(y_obs), y_obs.sum()

# Grid
theta_grid = np.linspace(0.001, 0.999, 1000)

# Non-standard prior: p(theta) ∝ exp(sin(π*theta²)) (non-conjugate)
log_prior = np.sin(np.pi * theta_grid**2)
log_likelihood = n_clicks * np.log(theta_grid) + (N - n_clicks) * np.log(1 - theta_grid)

log_unnorm_post = log_prior + log_likelihood
log_unnorm_post -= log_unnorm_post.max()  # numerical stability
unnorm_post = np.exp(log_unnorm_post)
q = unnorm_post / (unnorm_post.sum() * (theta_grid[1] - theta_grid[0]))  # normalise

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(theta_grid, q, color='steelblue', lw=2, label='Grid posterior q(θ)')
ax.fill_between(theta_grid, q, alpha=0.2)
ax.set_xlabel('θ'); ax.set_ylabel('Density')
ax.set_title('Grid approximation for non-conjugate posterior')
ax.legend()
plt.tight_layout(); plt.show()

# Posterior mean and 95% credibility interval
dtheta = theta_grid[1] - theta_grid[0]
post_mean = np.sum(theta_grid * q) * dtheta
cdf = np.cumsum(q) * dtheta
l95 = theta_grid[np.searchsorted(cdf, 0.025)]
u95 = theta_grid[np.searchsorted(cdf, 0.975)]
print(f'Posterior mean:          {post_mean:.3f}')
print(f'95% credibility interval: [{l95:.3f}, {u95:.3f}]')

<a id='week3'></a>

<div class="alert alert-block alert-info">

### **Week 3 | bayesian classification & laplace approximations**

Bayesian linear regression, posterior predictive distribution, hyperparameters

</div>

##### **3.1 Generative versus discriminative classification**


| Approach | Model | Pros | Cons |
|----------|-------|------|------|
| **Generative** | $p(x|y)$ and $p(y)$ → use Bayes to get $p(y|x)$ | Can handle missing data, generate samples | Assumptions on $p(x|y)$ may be wrong |
| **Discriminative** | Model $p(y|x)$ directly | Often better calibrated; more flexible | Cannot handle missing inputs |


##### **3.2 Logistic regression (discriminative)**


$$y_n | w, x_n \sim \text{Ber}(\sigma(w^T \phi(x_n)))$$

where $\sigma(a) = \frac{1}{1+e^{-a}}$ is the **logistic sigmoid**.

**Prior:** $w \sim \mathcal{N}(0, \alpha^{-1}I)$

**Log-joint:**
$$\log p(y, w) = \sum_n [y_n \log \sigma(f_n) + (1-y_n)\log(1-\sigma(f_n))] - \frac{\alpha}{2}w^T w + \text{const}$$

The posterior $p(w|y)$ is **not Gaussian** (non-conjugate) — need approximate inference.


##### **3.3 The laplace approximation**

**Idea:** Approximate the posterior with a Gaussian centred at the MAP estimate.

**Steps:**
1. Find MAP: $\hat{w} = \arg\max_w \log p(y|w) + \log p(w)$  (use gradient ascent / scipy.minimize)
2. Compute Hessian: $H = \nabla^2 \log p(y,w)|_{w=\hat{w}}$
3. Approximate: $p(w|y) \approx q(w) = \mathcal{N}(w | \hat{w},\; S)$ where $S = (-H)^{-1}$

> **Why does this work?** A second-order Taylor expansion of $\log p(y,w)$ around the mode gives a quadratic (which corresponds to a Gaussian). $-H$ must be positive definite at a maximum.

**For logistic regression:**
$$H = -\Phi^T W \Phi - \alpha I$$
where $W = \text{diag}(\sigma(f_n)(1-\sigma(f_n)))$ is the diagonal matrix of second-derivative weights.



##### **3.4 Posterior predictive for classification**


With Laplace approx $p(w|y) \approx \mathcal{N}(w|\hat{w}, S)$:

$$p(y^*=1|y,x^*) = \int \sigma(w^T\phi(x^*))\,\mathcal{N}(w|\hat{w},S)\,dw$$

This is **analytically intractable**. Three strategies:
1. **Sampling:** Draw $w^{(s)} \sim q(w)$, estimate $p(y^*=1) \approx \frac{1}{S}\sum_s \sigma(w^{(s)T}\phi^*)$
2. **Numerical integration**
3. **Probit approximation** (see below)


##### **3.5 The probit approximation**



Approximate the sigmoid with the Gaussian CDF:
$$\sigma(a) \approx \Phi\!\left(\frac{a}{\sqrt{\pi/8}}\right)$$

Then the integral becomes tractable:
$$p(y^*=1|y,x^*) \approx \Phi\!\left(\frac{\mu_{f^*}}{\sqrt{1 + \frac{\pi}{8}\sigma^2_{f^*}}}\right)$$

where $\mu_{f^*} = \hat{w}^T\phi^*$ and $\sigma^2_{f^*} = (\phi^*)^T S \phi^*$.

> **Key property of probit approx:** The posterior predictive mean of $f^*$ is scaled down by the variance — uncertainty flattens the sigmoid towards 0.5.

In [ ]:
# Week 3 Demo: Laplace approximation for logistic regression

from scipy.optimize import minimize

np.random.seed(42)
# Synthetic binary data
N = 30
x = np.linspace(-3, 3, N)
w_true = np.array([0.5, 1.5])
Phi = np.column_stack([np.ones(N), x])   # design matrix [1, x]
p_true = sigmoid(Phi @ w_true)
y = (np.random.rand(N) < p_true).astype(float)

alpha = 1.0  # prior precision

def neg_log_joint(w):
    f = Phi @ w
    ll = np.sum(y * np.log(sigmoid(f) + 1e-10) + (1-y) * np.log(1 - sigmoid(f) + 1e-10))
    lp = -0.5 * alpha * w @ w
    return -(ll + lp)

res = minimize(neg_log_joint, np.zeros(2), method='BFGS')
w_map = res.x

# Hessian → posterior covariance
from scipy.optimize import approx_fprime
f_map = Phi @ w_map
pi = sigmoid(f_map)
W_diag = pi * (1 - pi)  # Bernoulli second derivative weights
H_neg = Phi.T @ np.diag(W_diag) @ Phi + alpha * np.eye(2)
S = np.linalg.inv(H_neg)  # posterior covariance

print(f'MAP estimate:      w0={w_map[0]:.3f}, w1={w_map[1]:.3f}')
print(f'Posterior std:     σ0={np.sqrt(S[0,0]):.3f}, σ1={np.sqrt(S[1,1]):.3f}')

# Predict using probit approximation
x_star = np.linspace(-4, 4, 200)
Phi_star = np.column_stack([np.ones_like(x_star), x_star])
mu_f = Phi_star @ w_map
var_f = np.diag(Phi_star @ S @ Phi_star.T)
p_pred_probit = norm.cdf(mu_f / np.sqrt(1 + np.pi/8 * var_f))

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(x, y, zorder=5, label='Data', color='k')
ax.plot(x_star, sigmoid(Phi_star @ w_map), 'r--', label='Plug-in (MAP)')
ax.plot(x_star, p_pred_probit, 'b-', lw=2, label='Probit approx.')
ax.axhline(0.5, color='gray', linestyle=':')
ax.set_xlabel('x'); ax.set_ylabel('p(y*=1|y,x*)')
ax.set_title('Laplace + Probit approximation for logistic regression')
ax.legend(); plt.tight_layout(); plt.show()

<a id='week4'></a>

<div class="alert alert-block alert-info">

### **Week 4 | Bayesian linear regression**

Bayesian versus classical datasets, bayesian methods for classification, bayesian logistic regression, laplace approximations, the posterior predictive distribution

</div>

##### **4.1 The model**


$$y_n = \underbrace{w^T \phi(x_n)}_{f(x_n)} + \epsilon_n, \quad \epsilon_n \sim \mathcal{N}(0, \sigma^2)$$

**Design matrix** $\Phi \in \mathbb{R}^{N \times D}$ where row $n$ is $\phi(x_n)^T$.


##### **4.2 The four key equations**



With prior $p(w) = \mathcal{N}(w|0, \alpha^{-1}I)$ and $\beta = 1/\sigma^2$:

| Distribution | Expression |
|---|---|
| **Prior** | $p(w) = \mathcal{N}(w\|0, \alpha^{-1}I)$ |
| **Likelihood** | $p(y\|w) = \mathcal{N}(y\|\Phi w, \sigma^2 I)$ |
| **Posterior** | $p(w\|y) = \mathcal{N}(w\|m, S)$ |
| **Marginal likelihood** | $p(y) = \mathcal{N}(y\|0, \sigma^2 I + \alpha^{-1}\Phi\Phi^T)$ |

**Posterior parameters (must memorise!):**
$$S = (\alpha I + \beta \Phi^T\Phi)^{-1}, \qquad m = \beta S \Phi^T y$$


##### **4.3 Predictive distributions**



For a new point $x^*$ with $\phi^* = \phi(x^*)$:

$$p(f^*|y, x^*) = \mathcal{N}(f^* | \underbrace{m^T\phi^*}_{\text{pred. mean}},\; \underbrace{(\phi^*)^T S \phi^*}_{\text{epistemic var.}})$$

$$p(y^*|y, x^*) = \mathcal{N}(y^* | m^T\phi^*,\; (\phi^*)^T S \phi^* + \underbrace{\sigma^2}_{\text{aleatoric}})$$

> **Why does variance decompose this way?**  
> $y^* = f^* + \epsilon^*$ and $f^*$, $\epsilon^*$ are independent, so $\mathbb{V}[y^*] = \mathbb{V}[f^*] + \mathbb{V}[\epsilon^*]$.


##### **4.4 MAP = ridge regression**

$$\hat{w}_{\text{MAP}} = \arg\max_w \log p(w|y) = (\alpha I + \beta\Phi^T\Phi)^{-1}\beta\Phi^T y = \arg\min_w \left\{\|y - \Phi w\|^2 + \frac{\alpha}{\beta}\|w\|^2\right\}$$

Ridge regularisation $\lambda = \alpha/\beta$ is a direct consequence of the Gaussian prior with precision $\alpha$.


##### **4.5 Model selection: marginal likelihood (evidence)**


$$\log p(y|\alpha, \beta) = -\frac{N}{2}\log(2\pi) - \frac{1}{2}\log|\sigma^2 I + \alpha^{-1}\Phi\Phi^T| - \frac{1}{2}y^T(\sigma^2 I + \alpha^{-1}\Phi\Phi^T)^{-1}y$$

**Why does marginal likelihood select good models?**  
It balances data fit against model complexity. Overly complex models spread their prior probability too thinly → low marginal likelihood.


##### **4.6 Deriving the posterior (completing the square)**

Strategy: express $\log p(w|y)$ as a quadratic in $w$, then match to the form of $\log\mathcal{N}(w|m,S)$:
$$\log \mathcal{N}(w|m,S) = -\frac{1}{2}w^T S^{-1} w + m^T S^{-1} w + \text{const}$$

**Equate second-order term:** $S^{-1} = \beta\Phi^T\Phi + \alpha I$  
**Equate first-order term:** $m^T S^{-1} = \beta y^T\Phi \Rightarrow m = \beta S \Phi^T y$

In [ ]:
# Week 4 demo: bayesian linear regression with uncertainty

np.random.seed(0)

# Data
N = 10
x_train = np.linspace(0, 1, N)
y_train = np.sin(2 * np.pi * x_train) + 0.2 * np.random.randn(N)

# Feature map: polynomial degree 3
degree = 3
def phi(x, deg=degree):
    return np.column_stack([x**d for d in range(deg+1)])

Phi_train = phi(x_train)
alpha, beta = 0.5, 25.0  # prior precision, noise precision

# Posterior
S_inv = alpha * np.eye(degree+1) + beta * Phi_train.T @ Phi_train
S = np.linalg.inv(S_inv)
m = beta * S @ Phi_train.T @ y_train

# Predictions
x_star = np.linspace(-0.1, 1.1, 200)
Phi_star = phi(x_star)
mu_f = Phi_star @ m
var_f = np.diag(Phi_star @ S @ Phi_star.T)
var_y = var_f + 1/beta
std_y = np.sqrt(var_y)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, label in zip(axes, ['Posterior predictive', 'Uncertainty decomposition']):
    ax.scatter(x_train, y_train, c='k', zorder=5, label='Data')
    ax.plot(x_star, mu_f, 'b-', lw=2, label='Mean $m^T\\phi^*$')
    ax.fill_between(x_star, mu_f - 2*std_y, mu_f + 2*std_y, alpha=0.2, color='blue', label='±2σ (total)')
    ax.fill_between(x_star, mu_f - 2*np.sqrt(var_f), mu_f + 2*np.sqrt(var_f), alpha=0.3, color='green', label='±2σ (epistemic)')
    ax.legend(fontsize=9); ax.set_title(label)

plt.suptitle('Bayesian Linear Regression: Polynomial Basis (degree 3)')
plt.tight_layout(); plt.show()
print('Green = epistemic uncertainty (reduces with more data)')
print('Blue  = total (epistemic + aleatoric = σ² noise)')

<a id='week5'></a>

<div class="alert alert-block alert-info">

### **Week 5 | From parameters to functions**

Prior distribution for function spaces, gaussian process regression, covariance functions, hyperparameters, marginal likelihood

</div>

##### **5.1 From parameters to functions**

Bayesian linear model: $f(x) = w^T\phi(x)$, put a prior on $w$.  
**Gaussian Process:** put a prior **directly on the function** $f$.

$$f(x) \sim \mathcal{GP}(m(x),\; k(x,x'))$$

A GP is completely specified by:
- **Mean function:** $m(x) = \mathbb{E}[f(x)]$ (usually $m(x) = 0$)
- **Covariance function (kernel):** $k(x,x') = \text{Cov}[f(x), f(x')]$

Any finite collection of function values is jointly Gaussian:
$$[f(x_1), \dots, f(x_N)]^T \sim \mathcal{N}(\mathbf{0}, K) \quad\text{where } K_{ij} = k(x_i, x_j)$$


##### **5.2 Common kernel functions**


| Kernel | Formula | Stationary? | Isotropic? | Smoothness |
|--------|---------|-------------|------------|------------|
| **Squared Exponential** (RBF) | $\kappa^2 \exp\!\left(-\frac{\|x-x'\|^2}{2\ell^2}\right)$ | yes | yes | Infinitely differentiable |
| **Matérn 1/2** | $\kappa^2 \exp\!\left(-\frac{\|x-x'\|}{\ell}\right)$ | yes | yes | Continuous, not differentiable |
| **Matérn 3/2** | $\kappa^2(1+\frac{\sqrt{3}\|x-x'\|}{\ell})\exp(\cdots)$ | yes | yes | Once differentiable |
| **Linear** | $\alpha^{-1}x^Tx'$ | no | no | - |
| **ARD** | $\kappa^2\exp(-\frac{1}{2}(x-x')^T L^{-1}(x-x'))$ | yes | no | Inf. differentiable |

**Hyperparameters:**
- $\kappa > 0$: **magnitude** — controls the vertical scale of functions
- $\ell > 0$: **lengthscale** — controls how quickly functions vary (small $\ell$ = wiggly, large $\ell$ = smooth)

> **Stationary:** $k(x,x')$ depends only on $x - x'$.  
> **Isotropic:** $k(x,x')$ depends only on $\|x - x'\|$.


##### **5.3 GP regression - key equations**

Model: $y_n = f(x_n) + \epsilon_n$, $\epsilon_n \sim \mathcal{N}(0,\sigma^2)$

**Joint distribution:**
$$p(y, y^*) = \mathcal{N}\!\left(\begin{bmatrix}y\\y^*\end{bmatrix}\bigg| 0,\; \begin{bmatrix}K + \sigma^2 I & k_*\\ k_*^T & c\end{bmatrix}\right)$$

where $k_* = [k(x^*, x_n)]$ and $c = k(x^*,x^*) + \sigma^2$.

**Posterior predictive** (via Gaussian conditioning formula):
$$\boxed{p(y^*|y, x^*) = \mathcal{N}(y^* | \mu_{y^*|y},\; \sigma^2_{y^*|y})}$$

$$\mu_{y^*|y} = k_*(K + \sigma^2 I)^{-1} y$$
$$\sigma^2_{y^*|y} = c - k_*(K + \sigma^2 I)^{-1} k_*^T$$


##### **5.4 Gaussian conditioning formula (general)**


For jointly Gaussian $(y_1, y_2)$ with mean $\mu$ and covariance $\Sigma$:
$$p(y_1|y_2) = \mathcal{N}(y_1 | \mu_{1|2}, \Sigma_{1|2})$$
$$\mu_{1|2} = \mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(y_2 - \mu_2)$$
$$\Sigma_{1|2} = \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$$


##### **5.5 Marginal likelihood for hyperparameter selection**


$$\log p(y|\theta) = -\frac{N}{2}\log(2\pi) - \frac{1}{2}\log|K+\sigma^2 I| - \frac{1}{2}y^T(K+\sigma^2 I)^{-1}y$$

Optimise this numerically w.r.t. $\theta = \{\kappa, \ell, \sigma\}$ — much more efficient than cross-validation.


##### **5.6 Computational considerations**



- Matrix inversion $(K + \sigma^2 I)^{-1}$: $\mathcal{O}(N^3)$ — cubic cost!
- Memory: $\mathcal{O}(N^2)$
- **Cholesky decomposition** is used in practice to avoid direct inversion and compute log-determinants stably.

In [ ]:
# Week 5 demo: gaussian process regression

def squared_exp_kernel(x1, x2, kappa=1.0, ell=1.0):
    """Squared exponential (RBF) kernel."""
    x1, x2 = np.atleast_1d(x1), np.atleast_1d(x2)
    diff = x1[:, None] - x2[None, :]
    return kappa**2 * np.exp(-diff**2 / (2 * ell**2))

def gp_predict(x_train, y_train, x_star, kappa=1.0, ell=1.0, sigma=0.1):
    """GP posterior predictive mean and variance."""
    K    = squared_exp_kernel(x_train, x_train, kappa, ell) + sigma**2 * np.eye(len(x_train))
    k_s  = squared_exp_kernel(x_star, x_train, kappa, ell)
    c    = squared_exp_kernel(x_star, x_star, kappa, ell) + sigma**2 * np.eye(len(x_star))
    L    = np.linalg.cholesky(K)
    alpha_vec = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    mu   = k_s @ alpha_vec
    v    = np.linalg.solve(L, k_s.T)
    var  = np.diag(c - v.T @ v)
    return mu, var

# Training data
np.random.seed(1)
x_train = np.array([-2, -1, 0, 1, 2.5])
y_train = np.sin(x_train) + 0.1 * np.random.randn(len(x_train))
x_star  = np.linspace(-4, 4, 200)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
params = [(1.0, 0.5), (1.0, 1.0), (1.0, 2.0)]

for ax, (kappa, ell) in zip(axes, params):
    mu, var = gp_predict(x_train, y_train, x_star, kappa=kappa, ell=ell, sigma=0.1)
    std = np.sqrt(var)
    ax.plot(x_star, mu, 'b-', lw=2, label='Mean')
    ax.fill_between(x_star, mu-2*std, mu+2*std, alpha=0.3, color='blue', label='±2σ')
    ax.scatter(x_train, y_train, c='k', zorder=5, label='Data')
    ax.set_title(f'κ={kappa}, ℓ={ell}')
    ax.legend(fontsize=8)

fig.suptitle('GP Regression: Effect of lengthscale ℓ', fontweight='bold')
plt.tight_layout(); plt.show()

<a id='week6'></a>

<div class="alert alert-block alert-info">

### **Week 6 | GP classification & advanced kernels**

Covariance functions, gaussian processes in practice, gaussian process classification, neural networks for probabilistic modelling
</div>

##### **6.1 Gaussian process classification**


Model:
$$y_n | f_n \sim \text{Ber}(\sigma(f_n)), \qquad f \sim \mathcal{GP}(0, k(x,x'))$$

Unlike regression, the likelihood $p(y|f)$ is non-Gaussian (Bernoulli), so the posterior $p(f|y)$ is **not Gaussian** and is **intractable**.


##### **6.2 Laplace approximation for GP classification**




**Step 1:** Find MAP estimate $\hat{f}$ by maximising log-joint:
$$\log p(y,f) = \log p(y|f) + \log p(f) = \sum_n [y_n \log \sigma(f_n) + (1-y_n)\log(1-\sigma(f_n))] - \frac{1}{2}f^T K^{-1} f$$

Gradient and Hessian:
$$\nabla_f \log p(y,f) = g - K^{-1}f, \quad g_n = y_n - \sigma(f_n)$$
$$\nabla_f^2 \log p(y,f) = -\Lambda - K^{-1}, \quad \Lambda_{nn} = \sigma(f_n)(1-\sigma(f_n))$$

**Step 2:** Laplace approximation:
$$q(f) = \mathcal{N}(f | \hat{f},\; S), \quad S = (K^{-1} + \Lambda)^{-1}$$

**Step 3:** Posterior predictive for $f^*$:
$$p(f^*|y,x^*) \approx \mathcal{N}(f^* | \mu_{f^*},\; \sigma^2_{f^*})$$
$$\mu_{f^*} = k_*^T K^{-1}\hat{f}, \qquad \sigma^2_{f^*} = k(x^*,x^*) - k_*^T(K + \Lambda^{-1})^{-1}k_*$$

**Step 4:** Posterior predictive class probability via probit:
$$p(y^*=1|y,x^*) \approx \Phi\!\left(\frac{\mu_{f^*}}{\sqrt{1 + \frac{\pi}{8}\sigma^2_{f^*}}}\right)$$


##### **6.3 Constructing valid kernels**

If $k_1$ and $k_2$ are valid kernels, so are:
- $k_1 + k_2$ (sum)
- $k_1 \cdot k_2$ (product)
- $c \cdot k_1$ for $c > 0$ (scaling)

**Requirements:** must be symmetric and positive semi-definite.


##### **6.4 Kernels and feature spaces (Mercer's theorem)**





Every valid kernel $k(x,x')$ corresponds to an inner product in some (possibly infinite-dimensional) feature space $\phi(x)$:
$$k(x,x') = \phi(x)^T\phi(x')$$

The SE kernel has an **infinite-dimensional** implicit feature space.

In [ ]:
# Week 6 Demo: GP Classification with Laplace approximation

from scipy.optimize import minimize

def gp_laplace_classification(x_train, y_train, kappa=1.0, ell=1.0, n_iter=50):
    """Simple Laplace approximation for GP classification."""
    N = len(x_train)
    K = squared_exp_kernel(x_train, x_train, kappa, ell) + 1e-6 * np.eye(N)
    
    def log_joint(f):
        sigma_f = sigmoid(f)
        ll = np.sum(y_train * np.log(sigma_f + 1e-12) + (1-y_train) * np.log(1-sigma_f+1e-12))
        lp = -0.5 * f @ np.linalg.solve(K, f)
        return -(ll + lp)
    
    res = minimize(log_joint, np.zeros(N), method='L-BFGS-B')
    f_map = res.x
    
    # Hessian
    pi = sigmoid(f_map)
    Lambda = np.diag(pi * (1 - pi))
    S = np.linalg.inv(np.linalg.inv(K) + Lambda)
    
    return f_map, S, K

# Data
np.random.seed(7)
x_train = np.array([-2, -1, 0, 1, 2, 3])
y_train = np.array([0, 0, 0, 1, 1, 1], dtype=float)

f_map, S, K = gp_laplace_classification(x_train, y_train, kappa=1.0, ell=1.5)
x_star = np.linspace(-4, 5, 200)

# Predictive for f*
k_s = squared_exp_kernel(x_star, x_train, kappa=1.0, ell=1.5)
K_ss = squared_exp_kernel(x_star, x_star, kappa=1.0, ell=1.5)
mu_fstar = k_s @ np.linalg.solve(K, f_map)
pi_map = sigmoid(f_map)
Lambda = np.diag(pi_map * (1-pi_map))
var_fstar = np.diag(K_ss - k_s @ np.linalg.solve(K + np.linalg.inv(Lambda), k_s.T))

# Probit approximation
p_pred = norm.cdf(mu_fstar / np.sqrt(1 + np.pi/8 * var_fstar))

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(x_train, y_train, c=['r' if y==0 else 'b' for y in y_train], zorder=5, s=80)
ax.plot(x_star, p_pred, 'b-', lw=2, label='p(y*=1|y,x*) probit')
ax.fill_between(x_star, p_pred - 0.1*np.sqrt(var_fstar), p_pred + 0.1*np.sqrt(var_fstar),
                alpha=0.2, color='blue', label='Uncertainty')
ax.axhline(0.5, color='gray', linestyle='--')
ax.set_xlabel('x'); ax.set_ylabel('p(y*=1)')
ax.set_title('GP Classification: Laplace + Probit approximation')
ax.legend(); plt.tight_layout(); plt.show()

<a id='week7'></a>

<div class="alert alert-block alert-info">

### **Week 7 | Multi-class classification, decision thery & calibration**

Gaussan processes and neural networks, generalized linear models and non-gaussian likelihoods, generalization, evaluation, decision theory, calibration

</div>

##### **7.1 Softmax for multi-class classification**




For $K$ classes with $f_k(x) = w_k^T \phi(x)$:
$$p(y=k|x) = \text{softmax}_k(f(x)) = \frac{e^{f_k(x)}}{\sum_{j=1}^K e^{f_j(x)}}$$


##### **7.2 Uncertainty decomposition**




For a categorical predictive distribution $\pi_k = p(y^*=k|y,x^*)$:

| Measure | Formula | Meaning |
|---------|---------|----------|
| **Confidence** | $C = \max_k \pi_k$ | How sure is the model? |
| **Total entropy** | $H = -\sum_k \pi_k \log \pi_k$ | Total uncertainty |
| **Aleatoric** | $\mathbb{E}_{p(w|y)}[H(y^*|w,x^*)]$ | Inherent uncertainty |
| **Epistemic** | $H_{\text{total}} - H_{\text{aleatoric}}$ | Model uncertainty |

Max entropy = $\log K$ (uniform), achieved when model is maximally uncertain.


##### **7.3 Bayesian decision theory**


**Setup:**
- Actions: $\hat{y} \in \{1,\dots,K\}$
- Utility function: $U(y, \hat{y})$ = gain for predicting $\hat{y}$ when truth is $y$
- (Or equivalently, loss $L(y,\hat{y}) = -U(y,\hat{y})$)

**Optimal decision** = maximise expected utility:
$$\hat{y}^* = \arg\max_{\hat{y}} \mathbb{E}_{p(y^*|y,x^*)}[U(y^*, \hat{y})] = \arg\max_{\hat{y}} \sum_k \pi_k U(k, \hat{y})$$

**Special case — 0/1 loss** (misclassification rate):
$$U(y,\hat{y}) = \mathbb{1}[y = \hat{y}] \Rightarrow \hat{y}^* = \arg\max_k \pi_k$$

So picking the most probable class is optimal under 0/1 loss.

**Reject option:** Abstain if $C = \max_k \pi_k < \theta_{\text{reject}}$.


##### **7.4 Decision theory for regression**

| Loss function | Optimal predictor |
|---|---|
| Squared loss $(y - \hat{y})^2$ | $\hat{y} = \mathbb{E}[y^*|x^*]$ (posterior mean) |
| Absolute loss $|y - \hat{y}|$ | $\hat{y} = \text{median}(p(y^*|x^*))$ |
| 0/1 loss | $\hat{y} = \text{mode}(p(y^*|x^*))$ (MAP) |


##### **7.5 Model calibration**


A model is **well-calibrated** if predicted probabilities match empirical frequencies:
$$P(\text{correct} | C = p) = p \quad \forall p$$

**Expected Calibration Error (ECE):**
$$\text{ECE} = \sum_{b=1}^B \frac{|B_b|}{N}|\text{acc}(B_b) - \text{conf}(B_b)|$$

Visualised as a **reliability diagram** (calibration plot): confidence on x-axis, accuracy on y-axis. Perfect calibration = diagonal.

**Overconfident:** curve below diagonal.  
**Underconfident:** curve above diagonal.

In [ ]:
# Week 7 demo: bayesian decision theory

print('=== Binary classification: Cancer detection example ===')
print()

# Utility matrix
# U[y, y_hat]: row = true class, col = decision
#              no cancer   cancer
# no cancer:     1          -10
# cancer:       -100          1
U = np.array([[1, -10], [-100, 1]])

# Posterior predictive probability
p_cancer = 0.129  # from exam 2024
pi = np.array([1 - p_cancer, p_cancer])

print(f'Posterior predictive: P(cancer|x*) = {p_cancer:.3f}')
print()

for y_hat, action in enumerate(['Predict no cancer', 'Predict cancer']):
    expected_util = np.sum(pi * U[:, y_hat])
    print(f'  E[U(y*, ŷ={y_hat})] = {expected_util:.3f}  ({action})')

best = np.argmax([np.sum(pi * U[:, 0]), np.sum(pi * U[:, 1])])
print(f'\nOptimal decision: ŷ* = {best} ({["No cancer", "Cancer"][best]})')
print()
print('Intuition: even with low P(cancer)=0.129, the asymmetric utility')
print('(missing cancer has cost 100) pushes us to predict cancer.')

<a id='week8'></a>

<div class="alert alert-block alert-info">

### **Week 8 | monte carlo & metropolis-hasting**

Calibration, monte carlo methods, simple sampling methods, markov chain monte carlo methods

</div>

##### **8.1 Why we need sampling**



For most models, the posterior $p(\theta|y)$ is **analytically intractable**. MCMC methods generate (correlated) samples from $p(\theta|y)$ without knowing the normalisation constant.


##### **8.2 Monte carlo integration**



Goal: estimate $\bar{f} = \mathbb{E}_p[f(\theta)] = \int f(\theta)p(\theta)d\theta$

If we have i.i.d. samples $\theta^{(1)},\dots,\theta^{(S)} \sim p(\theta)$:
$$\hat{f} = \frac{1}{S}\sum_{i=1}^S f(\theta^{(i)})$$

**Properties:**
- **Unbiased:** $\mathbb{E}[\hat{f}] = \bar{f}$
- **Variance:** $\mathbb{V}[\hat{f}] = \frac{1}{S}\mathbb{V}[f(\theta)]$ — decreases as $1/S$
- **Does not suffer from curse of dimensionality** (unlike numerical quadrature)

**Many quantities are expectations:**
- Mean: $f(\theta) = \theta$
- Variance: $f(\theta) = (\theta - \mathbb{E}[\theta])^2$  
- Probabilities: $f(\theta) = \mathbb{1}[\theta > \tau]$
- Predictive density: $f(\theta) = p(y^*|\theta, x^*)$


##### **8.3 The metropolis-hastings algorithm**


**Key insight:** We only need to evaluate $p(\theta|y)$ up to a constant, because:
$$A_k = \min\!\left(1, \frac{p(\theta^*|y)\,q(\theta^{k-1}|\theta^*)}{p(\theta^{k-1}|y)\,q(\theta^*|\theta^{k-1})}\right) = \min\!\left(1, \frac{p(y,\theta^*)\,q(\theta^{k-1}|\theta^*)}{p(y,\theta^{k-1})\,q(\theta^*|\theta^{k-1})}\right)$$

The normalising constant $p(y)$ cancels!

**Algorithm:**
```
initialise θ⁰
for k = 1 to K:
    1. Propose: θ* ~ q(θ*|θ^(k-1))
    2. Compute: A_k = min(1, p(θ*|y)q(θ^(k-1)|θ*) / (p(θ^(k-1)|y)q(θ*|θ^(k-1))))
    3. Accept: θ^k = θ* with prob A_k, else θ^k = θ^(k-1)
```

**For symmetric proposals** (e.g. Gaussian): $q(a|b) = q(b|a)$, simplifies to:
$$A_k = \min\!\left(1, \frac{p(\theta^*|y)}{p(\theta^{k-1}|y)}\right)$$

This is the **Metropolis algorithm** (simpler version).


##### **8.4 Practical considerations**




| Issue | Solution |
|-------|----------|
| Burn-in / warm-up | Discard first $K_{\text{warmup}}$ samples |
| Proposal variance $\tau$ | Too small → slow mixing; too large → low acceptance. Rule of thumb: ~23% acceptance in high-D |
| Multiple chains | Run from different initialisations, check convergence |
| Correlated samples | MCMC samples are correlated → effective sample size $< S$ |

In [ ]:
# Week 8 demo: metropolis-hastings sampler

def metropolis(log_target, theta_init, tau, n_iter, seed=42):
    """Metropolis algorithm with isotropic Gaussian proposal."""
    rng = np.random.default_rng(seed)
    D = len(theta_init)
    samples = np.zeros((n_iter, D))
    samples[0] = theta_init
    n_accept = 0
    
    for k in range(1, n_iter):
        theta_current = samples[k-1]
        theta_star    = theta_current + tau * rng.standard_normal(D)
        
        log_ratio = log_target(theta_star) - log_target(theta_current)
        if np.log(rng.uniform()) < log_ratio:
            samples[k] = theta_star
            n_accept += 1
        else:
            samples[k] = theta_current
    
    acceptance_rate = n_accept / n_iter
    return samples, acceptance_rate

# Target: 2D banana-shaped distribution (from 2024 exam!)
def log_target_banana(theta):
    z1, z2 = theta
    return -(1-z1)**2 - 20*(z2 - z1**2)**2 - z1**2 - z2**2

n_iter = 5000
warmup = 1000

samples1, ar1 = metropolis(log_target_banana, np.array([0.0, 1.5]), tau=0.3, n_iter=n_iter, seed=0)
samples2, ar2 = metropolis(log_target_banana, np.array([-1.5, -0.5]), tau=0.3, n_iter=n_iter, seed=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Contour plot
z1g, z2g = np.meshgrid(np.linspace(-2,2,100), np.linspace(-1,3,100))
log_density = np.array([[log_target_banana([z1,z2]) for z1 in z1g[0]] for z2 in z2g[:,0]])
axes[0].contour(z1g, z2g, np.exp(log_density - log_density.max()), levels=10)
axes[0].scatter(*samples1[warmup:].T, alpha=0.1, s=2, c='blue', label='Chain 1')
axes[0].scatter(*samples2[warmup:].T, alpha=0.1, s=2, c='orange', label='Chain 2')
axes[0].legend(); axes[0].set_title('Posterior contours + MCMC samples')
axes[0].set_xlabel('z₁'); axes[0].set_ylabel('z₂')

# Trace plots
axes[1].plot(samples1[:, 0], lw=0.5, alpha=0.7, label='Chain 1')
axes[1].plot(samples2[:, 0], lw=0.5, alpha=0.7, label='Chain 2')
axes[1].axvline(warmup, color='r', linestyle='--', label='Warmup end')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('z₁')
axes[1].set_title('Trace plot for z₁'); axes[1].legend()

# Histogram after warmup
all_samples = np.vstack([samples1[warmup:], samples2[warmup:]])
axes[2].hist2d(all_samples[:,0], all_samples[:,1], bins=40, density=True)
axes[2].set_xlabel('z₁'); axes[2].set_ylabel('z₂')
axes[2].set_title('2D histogram of samples')

print(f'Acceptance rate chain 1: {ar1:.2f}')
print(f'Acceptance rate chain 2: {ar2:.2f}')
plt.tight_layout(); plt.show()

<a id='week9'></a>

<div class="alert alert-block alert-info">

### **Week 9**


</div>

##### **9.1 MCMC theory**


A Markov chain has a **stationary distribution** $p^*(\theta)$ if:
$$p^*(\theta) = \int T(\theta|\theta') p^*(\theta') d\theta'$$

For a chain to converge to $p^*(\theta)$, it must be:
- **(A1) Irreducible:** All states are reachable
- **(A2) Aperiodic:** No deterministic cycles
- **(A3) Positive recurrent:** Returns to any state with positive probability

**Detailed balance** (sufficient for $p^*$ to be stationary):
$$T(\theta'|\theta)\,p^*(\theta) = T(\theta|\theta')\,p^*(\theta')$$

MH with reasonable proposals satisfies detailed balance.


##### **9.2 Hamiltonian monte carlo (HMC)**





**Motivation:** Random walk MH has slow mixing in high-D. HMC uses **gradient information** to make directed proposals.

**Augmented system:** Introduce momentum $\nu \sim \mathcal{N}(0,I)$, define Hamiltonian:
$$H(\theta, \nu) = \underbrace{-\log p(\theta|y)}_{\text{potential energy}} + \underbrace{\frac{1}{2}\nu^T\nu}_{\text{kinetic energy}}$$

**Leapfrog integration** (numerically solve Hamiltonian dynamics):
$$\nu_{t+\eta/2} = \nu_t + \frac{\eta}{2}\nabla_\theta \log p(\theta_t|y)$$
$$\theta_{t+\eta} = \theta_t + \eta\, \nu_{t+\eta/2}$$
$$\nu_{t+\eta} = \nu_{t+\eta/2} + \frac{\eta}{2}\nabla_\theta \log p(\theta_{t+\eta}|y)$$

**HMC algorithm:** repeat for $L$ leapfrog steps, then MH accept/reject.

> **Why HMC is better:** Proposals follow the curvature of the posterior, so they have much higher acceptance rates and can traverse the space in fewer steps.


##### **9.3 Convergence diagnostics**


$\hat{R}$ statistic (Potential Scale Reduction Factor)

Run $M$ chains, each of length $S$. Let $B$ = between-chain variance, $W$ = within-chain variance:

$$\hat{R}^2 = \frac{S-1}{S} + \frac{1}{S}\frac{B}{W}$$

- $\hat{R} = 1$: chains have mixed perfectly  
- $\hat{R} > 1$: chains have not converged  
- **Rule of thumb:** $\hat{R} < 1.1$ (or $< 1.01$ for critical applications)

Effective Sample Size (ESS)

MCMC samples are correlated — $S$ correlated samples carry less information than $S$ i.i.d. samples:

$$S_{\text{eff}} = \frac{S}{1 + 2\sum_{t=1}^\infty \rho_t}$$

where $\rho_t$ is the autocorrelation at lag $t$.

Monte Carlo Standard Error (MCSE)

$$\text{MCSE} = \frac{\text{sd}(f(\theta))}{\sqrt{S_{\text{eff}}}}$$


##### **9.4 Pros and cons of MCMC methods**


| Method | Pros | Cons |
|--------|------|------|
| **MH** | Easy to implement, strong guarantees | Slow mixing, acceptance rate issues |
| **HMC** | Uses gradients → efficient, high-D friendly | Need gradient; $\eta$, $L$ tuning required |
| **NUTS** | Adaptive HMC, no tuning needed | Complex to implement |
| **Gibbs** | No acceptance step | Requires conditional distributions |

In [ ]:
# Week 9 demo: convergence diagnostics - R-hat and ESS

def compute_rhat(chains):
    """Compute R-hat statistic. chains: shape (n_chains, n_samples)"""
    M, S = chains.shape
    chain_means = chains.mean(axis=1)          # (M,)
    grand_mean  = chains.mean()
    B = S / (M - 1) * np.sum((chain_means - grand_mean)**2)   # between-chain variance
    W = np.mean(np.var(chains, axis=1, ddof=1))                # within-chain variance
    R2 = (S-1)/S + (1/S) * (B/W)
    return np.sqrt(R2)

def compute_ess(samples):
    """Compute effective sample size for a 1D chain."""
    S = len(samples)
    rho_sum = 0
    for t in range(1, S):
        rho_t = np.corrcoef(samples[:-t], samples[t:])[0, 1]
        if rho_t < 0.05:  # stop when autocorrelation is negligible
            break
        rho_sum += rho_t
    return S / (1 + 2 * rho_sum)

# Compare: well-mixed vs. poorly-mixed chains
np.random.seed(42)

# Well-mixed: multiple chains from same distribution
good_chains = np.array([norm.rvs(loc=2, scale=1, size=500, random_state=i) for i in range(4)])

# Poorly mixed: chains stuck in different modes
poor_chains = np.vstack([
    np.array([norm.rvs(loc=-3, scale=0.5, size=500, random_state=0),
              norm.rvs(loc=-3, scale=0.5, size=500, random_state=1)]),
    np.array([norm.rvs(loc=+3, scale=0.5, size=500, random_state=2),
              norm.rvs(loc=+3, scale=0.5, size=500, random_state=3)])
])

print('=== Convergence Diagnostics ===')
print(f'Well-mixed  chains → R-hat = {compute_rhat(good_chains):.3f}  (should be ≈1.0)')
print(f'Poorly-mixed chains → R-hat = {compute_rhat(poor_chains):.3f}  (should be >>1.0)')
print()

# ESS for correlated vs. uncorrelated samples
iid_samples  = norm.rvs(0, 1, size=1000, random_state=0)
# AR(1) correlated chain
rho_ar = 0.9
ar1_samples = np.zeros(1000)
for t in range(1, 1000):
    ar1_samples[t] = rho_ar * ar1_samples[t-1] + np.sqrt(1-rho_ar**2) * np.random.randn()

ess_iid = compute_ess(iid_samples)
ess_ar1 = compute_ess(ar1_samples)
print(f'ESS for i.i.d. samples (S=1000): {ess_iid:.0f}  (≈1000)')
print(f'ESS for correlated AR(1) chain:   {ess_ar1:.0f}  (<<1000)')

# Trace plots
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for i in range(4):
    axes[i//2, i%2].plot(good_chains[i], lw=0.5)
    axes[i//2, i%2].set_title(f'Well-mixed chain {i+1}')
    axes[i//2, i%2].set_xlabel('Iteration')
fig.suptitle('Trace plots for well-mixed chains', fontweight='bold')
plt.tight_layout(); plt.show()

<a id='cheatsheet'></a>

<div class="alert alert-block alert-info">

### **Exam cheat-sheet | key formula references**


</div>

##### **Gaussian distribution properties**



$$\mathcal{N}(x|\mu,\sigma^2): \quad \mathbb{E}[x] = \mu, \quad \mathbb{V}[x] = \sigma^2, \quad \text{Entropy} = \frac{1}{2}\log(2\pi e\,\sigma^2)$$

For a $D$-dimensional multivariate Gaussian $\mathcal{N}(x|\mu,\Sigma)$:
$$\text{Entropy} = \frac{D}{2}\log(2\pi e) + \frac{1}{2}\log|\Sigma| = \frac{1}{2}\log[(2\pi e)^D |\Sigma|]$$

For a **mean-field** Gaussian $q(w) = \prod_i \mathcal{N}(w_i|m_i,v_i)$:
$$\text{Entropy}(q) = \sum_i \frac{1}{2}\log(2\pi e\,v_i)$$


##### **Credibility intervals**


For $x \sim \mathcal{N}(\mu, \sigma^2)$, a 95% credibility interval:
$$[\mu - 1.96\sigma,\; \mu + 1.96\sigma]$$


##### **Bayesian linear regression (summary)**


$$S = (\alpha I + \beta \Phi^T\Phi)^{-1}, \quad m = \beta S\Phi^T y$$
$$p(y^*|y,x^*) = \mathcal{N}(y^* | m^T\phi^*,\; (\phi^*)^T S\phi^* + \sigma^2)$$


##### **Gaussian process regression (summary)**



$$\mu_{y^*|y} = k_*(K+\sigma^2 I)^{-1}y, \quad \sigma^2_{y^*|y} = c - k_*(K+\sigma^2 I)^{-1}k_*^T$$

where $k_* = [k(x^*,x_n)]_{n=1}^N$ and $c = k(x^*,x^*)+\sigma^2$.


##### **Laplace approximation summary**


$$q(\theta) = \mathcal{N}(\theta|\hat\theta, S), \quad \hat\theta = \arg\max \log p(y,\theta), \quad S = [-\nabla^2\log p(y,\theta)|_{\hat\theta}]^{-1}$$


##### **Probit approximation**



$$p(y^*=1|y,x^*) \approx \Phi\!\left(\frac{\mu_{f^*}}{\sqrt{1 + \frac{\pi}{8}\sigma^2_{f^*}}}\right)$$


##### **MH acceptance probability**



$$A_k = \min\!\left(1, \frac{p(\theta^*|y)\,q(\theta^{k-1}|\theta^*)}{p(\theta^{k-1}|y)\,q(\theta^*|\theta^{k-1})}\right)$$

For symmetric proposals: $A_k = \min(1, p(\theta^*|y)/p(\theta^{k-1}|y))$


##### **Grid approximation (2D exam example)**



From a grid $q(w_1, w_2)$ with non-zero entries at points $(w_1^{(i)}, w_2^{(j)})$:

$$\mathbb{E}[w_1] = \sum_{i,j} w_1^{(i)} \cdot q(w_1^{(i)}, w_2^{(j)})$$
$$\mathbb{E}[w_1^2] = \sum_{i,j} (w_1^{(i)})^2 \cdot q(w_1^{(i)}, w_2^{(j)})$$
$$\mathbb{V}[w_1] = \mathbb{E}[w_1^2] - (\mathbb{E}[w_1])^2$$


##### **Decision theory summary**



$$\hat{y}^* = \arg\max_{\hat{y}} \sum_k p(y^*=k|y,x^*)\cdot U(k, \hat{y})$$

For 0/1 loss: $\hat{y}^* = \arg\max_k p(y^*=k|y,x^*)$

##### **Convergence diagnostics**



$$\hat{R}^2 = \frac{S-1}{S} + \frac{1}{S}\frac{B}{W}, \quad \hat{R} < 1.1 \Rightarrow \text{converged}$$
$$S_{\text{eff}} = \frac{S}{1 + 2\sum_{t=1}^\infty \rho_t}$$

In [ ]:
# Cheat-sheet: useful python snippets for exam-style problems

from scipy.stats import norm

print('=== Gaussian credibility interval ===')
mu, sigma = 0.25, np.sqrt(6/5)
ci_95 = (mu - 1.96*sigma, mu + 1.96*sigma)
print(f'  N({mu}, {sigma**2:.2f}) → 95% CI = [{ci_95[0]:.4f}, {ci_95[1]:.4f}]')

print()
print('=== Entropy of Gaussian ===')
v1, v2 = 6/5, 6/5
H = 0.5 * np.log(2 * np.pi * np.e * v1) + 0.5 * np.log(2 * np.pi * np.e * v2)
print(f'  H(q*(θ)) = {H:.4f}  [for v1=v2={v1}]')

print()
print('=== Probit approximation ===')
mu_f, var_f = -0.5, 2.0
p_y1 = norm.cdf(mu_f / np.sqrt(1 + np.pi/8 * var_f))
print(f'  p(y*=1) via probit ≈ {p_y1:.4f}')

print()
print('=== Bayesian linear regression posterior ===')

In [ ]:
# Exam-style: given Φ, y, α, σ

x = np.array([0, 1, 2, 4, 5])
y = np.array([1.0, 0.5, -0.1, -0.9, 1.1])
Phi = np.column_stack([np.ones(5), x, x**2, x**3])
alpha = 1/2
sigma = 1/5
beta = 1/sigma**2

S_inv = alpha * np.eye(4) + beta * Phi.T @ Phi
S = np.linalg.inv(S_inv)
m = beta * S @ Phi.T @ y
print(f'  Posterior mean m = {m}')

In [ ]:
# Predictive at x*=3

x_star = 3
phi_star = np.array([1, x_star, x_star**2, x_star**3])
mu_pred = m @ phi_star
var_pred = phi_star @ S @ phi_star + sigma**2
print(f'  p(y*|y, x*=3) = N({mu_pred:.4f}, {var_pred:.4f})')

<div class="alert alert-block alert-info">

### **Common exam pitfalls & tips**


</div>


1. **Completing the square:** when asked for `p(w|y)`, always check if the log-posterior is quadratic in $w$ — if so it's Gaussian. Match $-\frac{1}{2}w^T S^{-1}w + m^T S^{-1}w$ to identify $S$ and $m$.

2. **Grid approximation sums must equal 1:** always verify $\sum_{i,j} q(w_i, w_j) = 1$ before computing moments.

3. **Probit vs. sigmoid:** for posterior predictive classification, use probit approximation unless instructed to use sampling. The formula is $\Phi(\mu_{f^*}/\sqrt{1+\frac{\pi}{8}\sigma^2_{f^*}})$.

4. **Prior predictive:** for GP, prior predictive is $p(y^*|x^*) = \mathcal{N}(0, k(x^*,x^*) + \sigma^2)$. For Bayesian LR: $p(y^*) = \mathcal{N}(0, (\phi^*)^T(\alpha^{-1}I)\phi^* + \sigma^2)$.

5. **MAP estimates:** MAP under Gaussian prior = ridge regression. MAP under Laplacian prior = LASSO.

6. **Switching from Q1 (mean-field) to Q2 (full-rank):** KL divergence can only decrease or stay the same when expanding the variational family, because Q1 ⊆ Q2.

7. **Scaling a kernel by c < 1:** for isnstance, $k_2 = \frac{1}{10}k_1$ → prior variance shrinks → posterior is pulled more toward the prior → predictive probability $p(y^*=1|y,x^*)$ is pulled toward 0.5.

8. **MH acceptance rate:** for the exam, compute in log-space: $\log A = \min(0, \log p(\theta^*) - \log p(\theta^{k-1}))$.

9. **ESS and R-hat:** always compute these for multi-chain MCMC experiments. $\hat{R} \approx 1$ and $S_{\text{eff}}$ large → good mixing.

10. **Report 2 decimal places** as instructed on the exam!